# Analiza strojenia i ewaluacji autoenkodera

Notebook zbiera wyniki do raportu końcowego (Fazy 5-9 specyfikacji). Czyta wyłącznie artefakty
wygenerowane przez `tune_autoencoder.py` i `evaluate_autoencoder.py` - nie trenuje niczego i nie potrzebuje GPU
ani danych.

**Wymagane pliki** (archiwum z serwera rozpakowane w katalogu repozytorium):

```
artifacts/tuning/
├── trials.csv, best_trial.json, best_config.yaml, param_importances.json, tuning.log
└── final_model/
    ├── training_history.json, experiment_config.yaml, model.pt
    └── test_evaluation/  (report.json, report.md, *.png)
```

**Ważne dla interpretacji:**
- Strojenie optymalizowało *recall fraudów na walidacji* przy progu dającym FPR < 1% na walidacji.
  Wszystkie liczby z części 1-3 pochodzą z walidacji.
- Zbiór testowy został użyty **jeden raz**, do końcowej oceny (część 4). Tylko te liczby odpowiadają na
  kryterium ze specyfikacji (recall ≥ 85% przy FPR < 1% na niezależnym zbiorze testowym).

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import Image, Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "fraud_detection").is_dir():
            return candidate
    raise FileNotFoundError("Uruchom notebook z katalogu repozytorium (lub jego podkatalogu)")


REPO_ROOT = find_repo_root(Path.cwd())
# zmienna środowiskowa pozwala wskazać inny katalog wyników bez edycji notebooka
TUNING = Path(os.environ.get("TUNING_ARTIFACTS", REPO_ROOT / "artifacts" / "tuning"))
FINAL_MODEL = TUNING / "final_model"
TEST_EVALUATION = FINAL_MODEL / "test_evaluation"

for required in (TUNING / "trials.csv", FINAL_MODEL / "training_history.json", TEST_EVALUATION / "report.json"):
    assert required.is_file(), f"Brak pliku {required} - rozpakuj archiwum z wynikami w katalogu repozytorium"

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")
plt.rcParams["figure.dpi"] = 110
print(f"Artefakty: {TUNING}")

## 1. Przebieg strojenia (Optuna)

Każda próba to pełny trening autoenkodera z wylosowanymi hiperparametrami (maks. 60 epok, early stopping).
Słabo rokujące próby były przerywane po kilku epokach (*pruning*, `MedianPruner`), a sampler TPE kierował
kolejne próby w obiecujące rejony przestrzeni.

In [ ]:
trials = pd.read_csv(TUNING / "trials.csv")
trials["minutes"] = pd.to_timedelta(trials["duration"]).dt.total_seconds() / 60
trials["hidden_dims"] = trials["user_attrs_hidden_dims"]
complete = trials[trials["state"] == "COMPLETE"].copy()
best_trial = json.loads((TUNING / "best_trial.json").read_text(encoding="utf-8"))

summary = trials.groupby("state").agg(prob=("number", "count"), minuty_lacznie=("minutes", "sum"))
display(summary)
print(f"Łącznie prób: {len(trials)}, czas obliczeń (suma po workerach): {trials['minutes'].sum() / 60:.1f} h")
print(f"Najlepsza próba #{best_trial['number']}: recall na walidacji {best_trial['validation_fraud_recall']:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(complete["number"], complete["value"] * 100, s=10, alpha=0.5, label="próba zakończona")
best_so_far = complete.sort_values("number")["value"].cummax() * 100
ax.plot(complete.sort_values("number")["number"], best_so_far, color="tab:red", label="najlepszy dotychczas")
ax.set_xlabel("numer próby")
ax.set_ylabel("recall na walidacji [%] (FPR < 1%)")
ax.set_title("Historia optymalizacji")
ax.legend()
plt.tight_layout()
plt.show()

**Do raportu:** krzywa „najlepszy dotychczas” szybko się wypłaszcza - po początkowej fazie kolejne setki prób
poprawiają wynik już tylko o pojedyncze fraudy (1 fraud na walidacji ≈ 0,42 p.p.). To argument, że
ograniczeniem nie jest dobór hiperparametrów.

## 2. Które hiperparametry miały znaczenie

Ważność (fANOVA/PED-ANOVA wbudowana w Optunę) mówi, jak bardzo dany parametr różnicował wyniki wszystkich
zakończonych prób - także bardzo słabych. Nie mówi, o ile parametr podnosi najlepszy wynik.

In [ ]:
importances = pd.Series(json.loads((TUNING / "param_importances.json").read_text(encoding="utf-8")))
fig, ax = plt.subplots(figsize=(8, 5))
importances.sort_values().plot.barh(ax=ax, color="tab:purple")
ax.set_xlabel("udział w zmienności wyniku")
ax.set_title("Ważność hiperparametrów")
plt.tight_layout()
plt.show()

## 3. Porównanie architektur i wariantów

**Uwaga metodologiczna:** TPE nie losuje równomiernie - częściej próbuje wariantów, które wcześniej
wypadały dobrze. Liczebności grup i ich średnie są przez to obciążone; uczciwszym porównaniem jest
**najlepszy wynik w grupie** (`max`), a różnice rzędu 1-2 p.p. mieszczą się w szumie walidacji (236 fraudów).

In [ ]:
def compare(column: str, label: str) -> pd.DataFrame:
    table = (
        complete.groupby(column)["value"]
        .agg(prob="count", srednia="mean", mediana="median", najlepszy="max")
        .sort_values("najlepszy", ascending=False)
    )
    table[["srednia", "mediana", "najlepszy"]] *= 100
    table.index.name = label
    return table.round(1)


for column, label in [
    ("params_hidden_layers", "liczba warstw ukrytych"),
    ("params_activation", "aktywacja"),
    ("params_denoising", "denoising"),
    ("params_optimizer", "optymalizator"),
    ("params_batch_norm", "batch norm"),
]:
    display(Markdown(f"**{label}** (recall na walidacji, %)"))
    display(compare(column, label))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, column, title in zip(
    axes,
    ["params_hidden_layers", "params_activation", "params_denoising"],
    ["liczba warstw ukrytych", "aktywacja", "denoising"],
):
    groups = [(str(key), group["value"] * 100) for key, group in complete.groupby(column)]
    ax.boxplot([values for _, values in groups])
    ax.set_xticks(range(1, len(groups) + 1), [name for name, _ in groups], rotation=20)
    ax.set_title(title)
    ax.set_ylabel("recall na walidacji [%]")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
scatter_specs = [
    ("user_attrs_parameters", "liczba parametrów modelu", True),
    ("params_latent_dim", "rozmiar bottlenecku (latent)", False),
    ("params_learning_rate", "learning rate", True),
]
for ax, (column, label, log_scale) in zip(axes, scatter_specs):
    ax.scatter(complete[column], complete["value"] * 100, s=10, alpha=0.5)
    if log_scale:
        ax.set_xscale("log")
    ax.set_xlabel(label)
    ax.set_ylabel("recall na walidacji [%]")
plt.tight_layout()
plt.show()

### Najlepsze konfiguracje

In [ ]:
top = complete.sort_values("value", ascending=False).head(15)
display(
    top[[
        "number", "value", "hidden_dims", "params_latent_dim", "params_activation", "params_batch_norm",
        "params_dropout", "params_denoising", "params_input_noise_std", "params_optimizer",
        "params_learning_rate", "params_batch_size", "user_attrs_parameters", "user_attrs_best_epoch", "minutes",
    ]].rename(columns=lambda name: name.replace("params_", "").replace("user_attrs_", ""))
)

### Pruning

Ile prób przerwano i w której epoce - pokazuje, ile czasu obliczeń zaoszczędziło przerywanie słabych prób.

In [ ]:
pruned = trials[trials["state"] == "PRUNED"]
print(f"Przerwane próby: {len(pruned)} z {len(trials)} ({len(pruned) / len(trials):.0%})")
if len(pruned):
    fig, ax = plt.subplots(figsize=(7, 3))
    pruned["user_attrs_pruned_at_epoch"].plot.hist(bins=30, ax=ax)
    ax.set_xlabel("epoka przerwania")
    ax.set_title("Kiedy przerywano słabe próby")
    plt.tight_layout()
    plt.show()

## 4. Model finalny

Zwycięska konfiguracja wytrenowana ponownie z pełnym budżetem epok (config bazowy). Wagi zapisane z epoki
o najniższym MSE walidacji (transakcje prawidłowe).

In [ ]:
final_config = yaml.safe_load((FINAL_MODEL / "experiment_config.yaml").read_text(encoding="utf-8"))
display(Markdown("**Konfiguracja modelu finalnego**"))
display(pd.DataFrame(
    [{"sekcja": section, "parametr": key, "wartosc": value}
     for section in ("model", "optimizer", "scheduler", "training")
     for key, value in final_config[section].items()]
).set_index(["sekcja", "parametr"]))

In [ ]:
history = json.loads((FINAL_MODEL / "training_history.json").read_text(encoding="utf-8"))
records = pd.DataFrame(history["records"])
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(records["epoch"], records["train_loss"], label="train MSE")
ax.plot(records["epoch"], records["validation_loss"], label="walidacja MSE (prawidłowe)")
ax.axvline(history["best_epoch"], color="black", linestyle=":", label=f"zapisana epoka {history['best_epoch']}")
ax.set_xlabel("epoka")
ax.set_ylabel("MSE")
recall_axis = ax.twinx()
recall_axis.plot(records["epoch"], records["validation_fraud_recall"] * 100, color="tab:red", alpha=0.6,
                 label="recall na walidacji")
recall_axis.set_ylabel("recall [%]")
ax.legend(loc="upper center")
recall_axis.legend(loc="upper right")
ax.set_title("Krzywe uczenia modelu finalnego")
plt.tight_layout()
plt.show()
print(f"Czas treningu: {history['total_seconds'] / 60:.1f} min, epok: {len(records)}, "
      f"early stopping: {history['stopped_early']}")

## 5. Ocena na zbiorze testowym (jednorazowa)

Próg wyznaczony na walidacji (najniższy próg z FPR < 1%) i zastosowany bez zmian na teście.
To jedyne liczby odpowiadające na kryterium ze specyfikacji.

In [ ]:
report = json.loads((TEST_EVALUATION / "report.json").read_text(encoding="utf-8"))
ae, base = report["autoencoder"], report["baseline"]
rows = ["recall", "fpr", "precision", "f1", "roc_auc", "pr_auc", "threshold", "validation_recall"]
display(pd.DataFrame({ae["name"]: [ae[key] for key in rows], base["name"]: [base[key] for key in rows]}, index=rows))

met = report["criteria"]["recall_met"] and report["criteria"]["fpr_met"]
display(Markdown(
    f"**Kryterium (recall ≥ {report['target_recall']:.0%} przy FPR < {report['max_false_positive_rate']:.0%}): "
    f"{'SPEŁNIONE' if met else 'NIESPEŁNIONE'}** - recall {ae['recall']:.1%} ({ae['tp']}/{ae['tp'] + ae['fn']}), "
    f"FPR {ae['fpr']:.2%}"
))

In [ ]:
def wilson_interval(successes: int, total: int, z: float = 1.96):
    share = successes / total
    centre = (share + z**2 / (2 * total)) / (1 + z**2 / total)
    margin = z * np.sqrt(share * (1 - share) / total + z**2 / (4 * total**2)) / (1 + z**2 / total)
    return centre - margin, centre + margin


low, high = wilson_interval(ae["tp"], ae["tp"] + ae["fn"])
needed = int(np.ceil(report["target_recall"] * (ae["tp"] + ae["fn"])))
print(f"Recall na teście: {ae['recall']:.1%}, 95% przedział ufności (Wilson): {low:.1%} - {high:.1%}")
print(f"Do celu {report['target_recall']:.0%} potrzeba {needed} wykrytych fraudów, wykryto {ae['tp']} "
      f"(różnica: {needed - ae['tp']})")

**Do raportu:** przedział ufności pokazuje, jak niepewny jest recall liczony na ~240 fraudach. Jeśli cel 85%
mieści się w przedziale, wynik nie różni się od celu w sposób statystycznie istotny - ale formalnie kryterium
nie jest spełnione i tak należy to przedstawić. Progu ani modelu nie wolno już zmieniać na podstawie testu.

In [ ]:
for name in ("confusion_matrices", "mse_distribution", "precision_recall"):
    display(Image(filename=str(TEST_EVALUATION / f"{name}.png")))

### Analiza niewykrytych fraudów (obowiązkowa próba przełamania)

In [ ]:
fn = report["false_negatives"]
print(f"Niewykryte fraudy: {fn['missed']} z {fn['missed'] + fn['detected']}; "
      f"nie wykrywa ich również Isolation Forest: {fn['missed_also_by_baseline']}")
print(f"Mediana MSE - niewykryte: {fn['median_mse_missed']:.3f}, wykryte: {fn['median_mse_detected']:.3f}, "
      f"prawidłowe: {fn['median_mse_legitimate']:.3f}")
display(Markdown("**Najtrudniejsze przypadki** (najniższe MSE wśród fraudów)"))
display(pd.DataFrame(fn["hardest_cases"]))
display(Markdown("**Cechy z największym błędem rekonstrukcji** (średni błąd kwadratowy)"))
display(pd.DataFrame(fn["feature_errors"]).set_index("feature"))

In [ ]:
timing = report["timing"]
display(Markdown("**Mierniki wydajnościowe**"))
display(pd.Series({
    "urządzenie": timing["device"],
    "trening autoenkodera [s]": round(timing["autoencoder_training_seconds"] or 0, 1),
    "predykcja autoenkodera na teście [s]": round(timing["autoencoder_test_prediction_seconds"], 3),
    "trening Isolation Forest [s]": round(timing["baseline_fit_seconds"], 2),
    "predykcja Isolation Forest na teście [s]": round(timing["baseline_test_prediction_seconds"], 3),
    "szczytowa pamięć GPU [MB]": timing["peak_gpu_memory_mb"],
    "czas strojenia (suma po workerach) [h]": round(trials["minutes"].sum() / 60, 1),
}, name="wartość").to_frame())

## 6. Punkty do wniosków (Faza 9)

- **Hipoteza** (AE trenowany na transakcjach prawidłowych daje wyższe MSE dla fraudów): porównaj mediany MSE
  wykrytych/niewykrytych fraudów i transakcji prawidłowych oraz histogram rozkładu MSE.
- **Kryterium ilościowe:** recall i FPR na teście wraz z przedziałem ufności (część 5).
- **Baseline:** różnica recall i PR-AUC względem Isolation Forest przy tym samym FPR.
- **Strojenie:** zakres przeszukanej przestrzeni (część 3), wypłaszczenie wyniku (część 1), najważniejsze
  hiperparametry (część 2).
- **Ograniczenia:** część fraudów ma błąd rekonstrukcji typowy dla transakcji prawidłowych i nie wykrywa ich
  również baseline; zbiór obejmuje tylko dwa dni i nie zawiera identyfikatora klienta ani historii transakcji;
  cechy V1-V28 po PCA nie są interpretowalne biznesowo.